# ETL of FanDuel Salary Data

* This notebook pulls in all the Fanduel salary data, with projections from FFA. 
* The Data is stored in JSON files, one for each week.
* The data comes from the FFA website.
* This notebook will combine all the data for the year, and overwrite the old file as new weeks occur

In [2]:
# Import libraries
import json
from pathlib import Path
import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

# Set base data folder (adjust if needed)
data_dir = Path(r"C:\Users\onlyu\OneDrive\Fantasy Football\data\FFA_DFS")

# Find JSON files for 2025 weekly data
fd_week_files = sorted(data_dir.glob("FD_2025_week_*.json"))
fda_optimal_files = sorted(data_dir.glob("FD_2025_FFA_optimal_week_*.json"))

print(f"Found {len(fd_week_files)} FanDuel week files")
print(f"Found {len(fda_optimal_files)} FFA optimal week files")


Found 15 FanDuel week files
Found 15 FFA optimal week files


## See which files are already in the folder

In [4]:
def parse_week(file_path: Path) -> int:
    import re
    m = re.search(r'week[_\s]?(\d+)', file_path.name, re.IGNORECASE)
    return int(m.group(1)) if m else None

def parse_year(file_path: Path) -> int:
    import re
    m = re.search(r'FD_(\d{4})_', file_path.name, re.IGNORECASE)
    return int(m.group(1)) if m else None

for file in fd_week_files:
    week_num = parse_week(file)
    year_num = parse_year(file)
    print(f"Week {week_num}: 'Year' {year_num}: {file.name}")

Week 1: 'Year' 2025: FD_2025_week_01.json
Week 2: 'Year' 2025: FD_2025_week_02.json
Week 3: 'Year' 2025: FD_2025_week_03.json
Week 4: 'Year' 2025: FD_2025_week_04.json
Week 5: 'Year' 2025: FD_2025_week_05.json
Week 6: 'Year' 2025: FD_2025_week_06.json
Week 7: 'Year' 2025: FD_2025_week_07.json
Week 8: 'Year' 2025: FD_2025_week_08.json
Week 9: 'Year' 2025: FD_2025_week_09.json
Week 10: 'Year' 2025: FD_2025_week_10.json
Week 11: 'Year' 2025: FD_2025_week_11.json
Week 12: 'Year' 2025: FD_2025_week_12.json
Week 13: 'Year' 2025: FD_2025_week_13.json
Week 14: 'Year' 2025: FD_2025_week_14.json
Week 15: 'Year' 2025: FD_2025_week_15.json


## Write function to read all the json files in the folder.

In [5]:
# Load and combine JSON files from the 'data' list only
FD_WEEK_COLUMNS = ['Rank','Lock', 'Exclude', 'Player', 'Team', 'Position', 'Salary [$]', 'Floor', 'Points', 'Ceiling', 'SD', 'Uncertainty', 'Tier','PPD']

def load_fd_week_df(file_path: Path) -> pd.DataFrame:
    with file_path.open("r", encoding="utf-8") as f:
        payload = json.load(f)


        # Handle both old format (single object) and new format (list of sections)
        all_rows = []
        if isinstance(payload, list):
            # New format: list of sections, each with "data" key
            for section in payload:
                rows = section.get("data", [])
                all_rows.extend(rows)
        else:
            # Old format: single object with "data" key
            all_rows = payload.get("data", [])
        
        df = pd.DataFrame(all_rows, columns=FD_WEEK_COLUMNS)
        df["week"] = parse_week(file_path)
        df["Year"] = parse_year(file_path)
        # Cast useful numeric columns
        #for col in ['Salary [$]', 'Floor', 'Points', 'Ceiling', 'SD', 'Uncertainty']:
        #    df[col] = pd.to_numeric(df[col], errors='coerce')
    return df

for file in fd_week_files:
    df = load_fd_week_df(file)
    print(f"Week {df['week'].iloc[0]} 'Year' {df['Year'].iloc[0]} rows: {len(df):,}")
    #display(df.head(3))

Week 1 'Year' 2025 rows: 25
Week 2 'Year' 2025 rows: 25
Week 3 'Year' 2025 rows: 25
Week 4 'Year' 2025 rows: 25
Week 5 'Year' 2025 rows: 25
Week 6 'Year' 2025 rows: 25
Week 7 'Year' 2025 rows: 25
Week 8 'Year' 2025 rows: 25
Week 9 'Year' 2025 rows: 25
Week 10 'Year' 2025 rows: 25
Week 11 'Year' 2025 rows: 25
Week 12 'Year' 2025 rows: 25
Week 13 'Year' 2025 rows: 250
Week 14 'Year' 2025 rows: 300
Week 15 'Year' 2025 rows: 250


## Save all files to a CSV

In [6]:

all_dfs = []
for fp in fd_week_files:
    try:
        all_dfs.append(load_fd_week_df(fp))
    except Exception as e:
        print(f"Error loading {fp.name}: {e}")

combined_df = pd.concat(all_dfs, ignore_index=True) if all_dfs else pd.DataFrame(columns=FD_WEEK_COLUMNS + ["week"] + ["Year"]) 
print(f"Combined rows: {len(combined_df):,}")

# Save combined data to CSV
output_csv = data_dir / "FD_combined_salary_data.csv"
combined_df.to_csv(output_csv, index=False)
print(f"Saved combined data to {output_csv}")

Combined rows: 1,100
Saved combined data to C:\Users\onlyu\OneDrive\Fantasy Football\data\FFA_DFS\FD_combined_salary_data.csv
